**Выполнил(а):** Михаил Осколков, Антон Бердников группа ИНБс-5301, вариант 2
Биграммы

## Шаг 0. Подготовка (как в части 1)

Папка `datasets` должна лежать рядом с блокнотом. В Google Colab загрузите датасет так же, как в части 1 — раскомментируйте нужный вариант в ячейке ниже (локально ничего делать не нужно).

In [ ]:
# --- Google Colab, вариант А: архив лежит в корне Google Диска ---
from google.colab import drive
drive.mount('/content/drive')
!apt-get install -y p7zip-full > /dev/null
!7z x -p1 -o/content/ "/content/drive/MyDrive/datasets.zip" -y

# --- Google Colab, вариант Б: загрузка архива в текущую сессию ---
# from google.colab import files
# uploaded = files.upload()          # выберите datasets.zip
# !apt-get install -y p7zip-full > /dev/null
# !7z x -p1 datasets.zip -y

Mounted at /content/drive

7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan /content/drive/MyDrive/                                 1 file, 39644509 bytes (38 MiB)

Extracting archive: /content/drive/MyDrive/datasets.zip
 40% 4096 Open              --
Path = /content/drive/MyDrive/datasets.zip
Type = zip
Physical Size = 39644509

  0%     10% 969         19% 1929          29% 2859          39% 3730          48% 4756          58% 5784          69% 6933          78% 7872 - datasets/trec07delay/data/inmail.8080                                                 

In [ ]:
import os
import email
from email import policy
import pandas as pd

DATA_DIR = 'datasets/trec07delay/data/'
LABELS_FILE = 'datasets/trec07delay/delay/index'

assert os.path.isdir(DATA_DIR) and os.path.isfile(LABELS_FILE), \
    'Датасет не найден — распакуйте datasets.zip рядом с блокнотом (шаг 0)'

def text_email(path):
    """Извлекает тему и текстовое (text/plain) тело письма — та же функция, что в части 1."""
    try:
        with open(path, 'rb') as f:
            msg = email.message_from_binary_file(f, policy=policy.default)
        subject = str(msg.get('Subject') or '')
        parts = []
        for part in msg.walk():
            if part.get_content_type() == 'text/plain':
                try:
                    content = part.get_content()
                    if isinstance(content, str):
                        parts.append(content)
                except Exception:
                    pass
        return subject + ' ' + ' '.join(parts)
    except Exception:
        return ''

# Читаем метки: имя файла → 'spam'/'ham'
labels = {}
with open(LABELS_FILE) as f:
    for line in f:
        parts = line.split()
        if len(parts) == 2:
            labels[parts[1].split('/')[-1]] = parts[0]

print('Меток прочитано:', len(labels))

Меток прочитано: 10000


In [ ]:
# Читаем все письма в список текстов X и список меток y (займёт до минуты)
X, y = [], []
empty = 0
filelist = sorted(labels, key=lambda name: int(name.split('.')[1]))
for filename in filelist:
    text = text_email(DATA_DIR + filename)
    if len(text.split()) <= 2:      # отсев пустых и HTML-only писем по сырому тексту;
        empty += 1                  # в части 1 он строже (после удаления стоп-слов), поэтому числа отличаются
        continue
    X.append(text)
    y.append(labels[filename])

print(f'Прочитано писем: {len(X)}, пропущено пустых: {empty}')
print('Пример текста (первые 200 символов):')
print(X[0][:200])

Прочитано писем: 9868, пропущено пустых: 132
Пример текста (первые 200 символов):
Generic Cialis, branded quality@  


## Шаг 1. Мешок слов: превращаем текст в числа

Модели машинного обучения работают с числами, а не с текстом. Самое простое числовое представление текста — **«мешок слов» (bag-of-words)**:

1. Проходим по всем документам и собираем **словарь** — список всех встретившихся слов.
2. Каждый документ представляем вектором длиной со словарь: на i-м месте стоит **число вхождений** i-го слова в этот документ.

Порядок слов и грамматика при этом теряются — остаётся только «какие слова и сколько раз встретились». Для классификации спама этого, как ни странно, почти достаточно. Разберём на простом примере из трёх «писем»:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

toy = ['hello mr bear',
       'hello hello gunter',
       'goodbye mr gunter']

toy_vectorizer = CountVectorizer()
toy_counts = toy_vectorizer.fit_transform(toy)

print('Словарь:', list(toy_vectorizer.get_feature_names_out()))
print(toy_counts.toarray())   # строки — письма, столбцы — слова словаря

Словарь: ['bear', 'goodbye', 'gunter', 'hello', 'mr']
[[1 0 0 1 1]
 [0 0 1 2 0]
 [0 1 1 0 1]]


Столбцы идут в алфавитном порядке словаря (`bear`, `goodbye`, `gunter`, `hello`, `mr`), поэтому первая строка `[1 0 0 1 1]` читается так: в письме A по одному разу встречаются `bear`, `hello` и `mr`, а `goodbye` и `gunter` — ни разу.

В реальном корпусе словарь содержит сотни тысяч слов, а в каждом отдельном письме встречается лишь малая их часть — матрица почти целиком нулевая. Поэтому `CountVectorizer` хранит её в **разреженном (sparse)** формате: в памяти лежат только ненулевые элементы.

In [ ]:
# Случайное разбиение 70/30; stratify=y сохраняет долю спама одинаковой в обеих частях
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=2, stratify=y)

print(f'train: {len(X_train)} писем, test: {len(X_test)} писем')
print('Доля спама: train {:.1%}, test {:.1%}'.format(
    sum(v == 'spam' for v in y_train) / len(y_train),
    sum(v == 'spam' for v in y_test) / len(y_test)))

train: 6907 писем, test: 2961 писем
Доля спама: train 80.3%, test 80.3%


In [ ]:
# Строим словарь ПО ТРЕНИРОВОЧНОЙ ВЫБОРКЕ и векторизуем обе части
vectorizer = CountVectorizer(ngram_range=(1, 2)) #добавил биграммы
X_train_counts = vectorizer.fit_transform(X_train)   # fit: составить словарь + посчитать слова
X_test_counts = vectorizer.transform(X_test)         # только посчитать — словарь уже зафиксирован

print('Размер словаря:', len(vectorizer.vocabulary_))
print('Матрица train:', X_train_counts.shape, '| ненулевых элементов:', X_train_counts.nnz)
print('Матрица test: ', X_test_counts.shape)

Размер словаря: 305934
Матрица train: (6907, 305934) | ненулевых элементов: 1145780
Матрица test:  (2961, 305934)


> **Важно:** `fit_transform` вызывается только на train — словарь строится по тренировочным данным. Тестовая выборка проходит только через `transform`: слова, которых не было в train, просто игнорируются. Иначе информация о тесте «протечёт» в обучение.

## Шаг 2. Наивный байесовский классификатор

Идея: для каждого класса (спам / ham) по тренировочной выборке оценивается, с какой вероятностью в нём встречается каждое слово словаря. Для нового письма по **формуле Байеса** вычисляется, к какому классу оно вероятнее относится, — по совокупности его слов.

Классификатор называется «наивным», потому что слова считаются независимыми друг от друга (в реальном языке это, конечно, не так). На практике для текстов метод работает на удивление хорошо, а обучается за доли секунды.

`MultinomialNB` — вариант наивного Байеса для счётчиков слов (наш случай). Чтобы слово, не встречавшееся в train, не обнулило всё произведение вероятностей, применяется **сглаживание Лапласа** (параметр `alpha`, по умолчанию 1.0). Подробная теория — в Приложении А методички.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train_counts, y_train)     # обучение: подсчёт вероятностей слов в каждом классе
y_pred = model.predict(X_test_counts)  # прогноз для тестовой выборки

print('Первые 10 прогнозов:      ', [str(v) for v in y_pred[:10]])
print('Первые 10 истинных меток: ', y_test[:10])

Первые 10 прогнозов:       ['spam', 'ham', 'spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'spam']
Первые 10 истинных меток:  ['spam', 'ham', 'spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'spam']


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

print(f'Accuracy: {accuracy_score(y_test, y_pred):.1%}\n')
print(classification_report(y_test, y_pred, digits=3))

cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=['spam', 'ham']),
    index=['на самом деле спам', 'на самом деле ham'],
    columns=['прогноз: спам', 'прогноз: ham']
)
print(cm)

base_metrics = {
    'Accuracy':         0.979,
    'Precision (spam)': 0.994,
    'Recall (spam)':    0.980,
    'F1 (spam)':        0.987,
    'Размер словаря':   45547,
}

var_metrics = {
    'Accuracy':         accuracy_score(y_test, y_pred),
    'Precision (spam)': precision_score(y_test, y_pred, pos_label='spam'),
    'Recall (spam)':    recall_score(y_test, y_pred, pos_label='spam'),
    'F1 (spam)':        f1_score(y_test, y_pred, pos_label='spam'),
    'Размер словаря':   len(vectorizer.get_feature_names_out()),
}

df_compare = pd.DataFrame({
    'Метрика':          list(base_metrics.keys()),
    'Базовый (1,1)':    list(base_metrics.values()),
    'Вариант 2 (1,2)':  list(var_metrics.values()),
})
df_compare['Разница'] = df_compare['Вариант 2 (1,2)'] - df_compare['Базовый (1,1)']

print("Сравнение вариантов")
print(f"Словарь вырос в {var_metrics['Размер словаря'] / base_metrics['Размер словаря']:.1f} раз\n")
print(df_compare.round(4).to_string(index=False))

Accuracy: 98.0%

              precision    recall  f1-score   support

         ham      0.927     0.978     0.952       584
        spam      0.994     0.981     0.988      2377

    accuracy                          0.980      2961
   macro avg      0.961     0.979     0.970      2961
weighted avg      0.981     0.980     0.981      2961

                    прогноз: спам  прогноз: ham
на самом деле спам           2332            45
на самом деле ham              13           571
Сравнение вариантов
Словарь вырос в 6.7 раз

         Метрика  Базовый (1,1)  Вариант 2 (1,2)     Разница
        Accuracy          0.979           0.9804      0.0014
Precision (spam)          0.994           0.9945      0.0005
   Recall (spam)          0.980           0.9811      0.0011
       F1 (spam)          0.987           0.9877      0.0007
  Размер словаря      45547.000      305934.0000 260387.0000


**Выводы** *(заполните сами, 3–5 предложений)*: чем наивный Байес принципиально отличается от чёрного списка? Почему он заметно точнее? В чём его слабые места — что произойдёт, если спамеры изменят лексику?

В варианте 2 я заменил `CountVectorizer()` на `CountVectorizer(ngram_range=(1, 2))`.  
Это добавило к отдельным словам пары соседних слов (биграммы).  

Что изменилось:
- Размер словаря вырос в 6.7 раз (с 45 547 до 305 934 признаков).
- Время векторизации и обучения увеличилось.
- Метрики (accuracy, precision, recall для спама) немного улучшились.

Почему так произошло:
Биграммы позволяют модели учитывать короткие фразы и контекст, например «click here» или «free money».  
Это немного помогает, потому что такие фразы это сильные спам-маркеры.  
Однако наивный Байес всё ещё считает все признаки независимыми, а биграммы частично дублируют информацию из униграмм (слово «free» уже есть как отдельный признак).  
Из-за этого рост качества обычно небольшой, а иногда метрики даже падают из-за переобучения на редких биграммах.

Итог:  
Биграммы это полезный приём инженерии признаков, но они не устраняют главный недостаток наивного Байеса это предположение о независимости слов.  
Для серьёзного улучшения нужны модели, которые учитывают взаимодействия признаков (логистическая регрессия, SVM, деревья, нейросети).